## 第 4 回 演習 Notebook  |  Grover アルゴリズムこのノートブックは講義中に並走で実行する。各セクションのスライド番号と連動している。### 使い方- 各セルは順に実行する (上から下へ)- 一部のセル (Slide 32) は自分で実装する箇所がある- 実行に時間がかかるセル (26 qubit 版) はスキップ可### 環境Qiskit 1.x + qiskit-aer。Colab で初回実行する場合は次のセルで pip install。

In [ ]:
!pip install qiskit qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.5 MB/s eta 0:00:00


In [ ]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

backend = AerSimulator()

def run_aer(qc, shots=1024):
    """AerSimulator で回路実行、観測 counts を返す"""
    qc_t = transpile(qc, backend, optimization_level=0)
    result = backend.run(qc_t, shots=shots).result()
    return result.get_counts()

print("環境準備完了")

環境準備完了


## Slide 6 連動  |  2qubit Grover デモの再現前週 (#3) で見た N=4, 解 |11⟩ の Grover を再現する。1 反復で解の状態 |11⟩ がほぼ 100% で観測されることを確認する。回路の構成:- フェーズ A: H で 4 状態を等しく重ね合わせ- フェーズ B: Oracle (CZ で |11⟩ に位相 -1 を付ける)- フェーズ C: Diffusion (平均反転、a' = 2μ - a)- フェーズ D: 測定

In [ ]:
def build_2qubit_grover():
    qc = QuantumCircuit(2, 2)
    # フェーズ A: 重ね合わせ
    qc.h([0, 1])
    # フェーズ B: Oracle (|11⟩ に位相 -1)
    qc.cz(0, 1)
    # フェーズ C: Diffusion
    qc.h([0, 1])
    qc.x([0, 1])
    qc.cz(0, 1)
    qc.x([0, 1])
    qc.h([0, 1])
    # 平均反転 a' = 2μ - a の符号と一致させる位相調整
    qc.global_phase = np.pi
    qc.measure([0, 1], [0, 1])
    return qc

qc_demo = build_2qubit_grover()
counts = run_aer(qc_demo, shots=1024)
print(f"観測結果: {counts}")
print(f"→ |11⟩ がほぼ 100% で観測される (1 反復で完成)")

## Slide 8 連動  |  振幅推移の可視化Step 0/1/2 の各時点で振幅を Statevector から取り出して棒グラフ化する。スライドの 3 段棒グラフと同じ振幅推移が再現される。期待される振幅:- Step 0 (初期): (0.5, 0.5, 0.5, 0.5)、平均 μ = 0.5- Step 1 (Oracle 後): (0.5, 0.5, 0.5, -0.5)、平均 μ = 0.25- Step 2 (Diffusion 後): (0, 0, 0, 1.0)、平均 μ = 0.25

In [ ]:
def get_amplitudes_at_step(step):
    """step=0: 初期, step=1: Oracle 後, step=2: Diffusion 後"""
    qc = QuantumCircuit(2)
    qc.h([0, 1])
    if step >= 1:
        qc.cz(0, 1)  # Oracle
    if step >= 2:
        # Diffusion
        qc.h([0, 1]); qc.x([0, 1]); qc.cz(0, 1); qc.x([0, 1]); qc.h([0, 1])
        qc.global_phase = np.pi  # 符号調整
    sv = Statevector.from_instruction(qc)
    return np.real(sv.data)

# 棒グラフで可視化
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
labels = ['|00⟩', '|01⟩', '|10⟩', '|11⟩']
titles = ['Step 0: 初期 (フェーズ A 後)',
          'Step 1: Oracle 後 (フェーズ B 後)',
          'Step 2: Diffusion 後 (フェーズ C 後)']
for step in range(3):
    amps = get_amplitudes_at_step(step)
    mean = np.mean(amps)
    colors = ['steelblue'] * 4
    if step >= 1:
        colors[3] = 'crimson'  # 解 |11⟩ を強調表示
    axes[step].bar(range(4), amps, color=colors, edgecolor='black')
    axes[step].axhline(mean, color='red', linestyle='--', linewidth=1.2, label=f'μ = {mean:.3f}')
    axes[step].set_xticks(range(4)); axes[step].set_xticklabels(labels)
    axes[step].set_ylim(-0.6, 1.1); axes[step].set_title(titles[step])
    axes[step].set_ylabel('振幅')
    axes[step].legend()
plt.tight_layout()
plt.show()

print()
for step in range(3):
    amps = get_amplitudes_at_step(step)
    print(f"Step {step}: amps = {np.round(amps, 4)}, μ = {np.mean(amps):.4f}")

## Slide 10 連動  |  平均反転を式で検証Slide 9 の手計算 `a' = 2μ - a` を Statevector で再現し、公式と実装が一致することを確認する。`a` は Step 1 (Oracle 後) の振幅、`μ` はその平均、`a'` が Step 2 の振幅。

In [ ]:
amps_step1 = get_amplitudes_at_step(1)
mu = np.mean(amps_step1)
amps_predicted = 2 * mu - amps_step1   # 公式 a' = 2μ - a
amps_actual = get_amplitudes_at_step(2)

print(f"Step 1 振幅:        {np.round(amps_step1, 4)}")
print(f"平均 μ:              {mu:.4f}")
print()
print(f"公式予測 (2μ - a):  {np.round(amps_predicted, 4)}")
print(f"実測:                {np.round(amps_actual, 4)}")
print()
assert np.allclose(amps_predicted, amps_actual, atol=1e-10), \
    "公式と実装が一致しません"
print(f"→ 公式と実装が一致 (Diffusion = 平均反転)")

## Slide 12 連動  |  Diffusion 回路の 3 段階実装Diffusion = H⊗ⁿ → X⊗ⁿ → CC..CZ → X⊗ⁿ → H⊗ⁿ の 3 段階で平均反転を実装する。2 qubit 版でも 9 qubit (ナイーブ版) でも同じ構造を使うため、共有可能な関数として定義する。

In [ ]:
def diffusion(qc, qubits):
    """Diffusion 回路 (平均反転の実装)
    1. H⊗ⁿ      : 基底変換 |s⟩ → |0⋯0⟩
    2. X⊗ⁿ      : |0⋯0⟩ ⇄ |1⋯1⟩
    3. CC..CZ   : |1⋯1⟩ にだけ位相 -1
    4. X⊗ⁿ      : 元に戻す
    5. H⊗ⁿ      : 基底を |s⟩ に戻す
    最後に位相 π を加算して a' = 2μ - a と符号を揃える (確率は不変)
    """
    qc.h(qubits)
    qc.x(qubits)
    if len(qubits) == 2:
        qc.cz(qubits[0], qubits[1])
    else:
        qc.h(qubits[-1])
        qc.mcx(qubits[:-1], qubits[-1])
        qc.h(qubits[-1])
    qc.x(qubits)
    qc.h(qubits)
    qc.global_phase += np.pi

print("diffusion() 定義完了")

## Slide 12 連動  |  H 基底変換の確認Diffusion の段階 1「H⊗ⁿ で `|s⟩` が `|0⋯0⟩` になる」を Statevector で確認する。`|s⟩` は全状態の等しい重ね合わせ。それに H を再びかけると `|00⟩` に写る。新しい基底では `|0⋯0⟩` が「平均」の役割を担う。

In [ ]:
# まず |s⟩ を作る
qc1 = QuantumCircuit(2)
qc1.h([0, 1])  # |00⟩ → |s⟩ = (1/2)(|00⟩ + |01⟩ + |10⟩ + |11⟩)
sv_s = Statevector.from_instruction(qc1)
print(f"|s⟩ = {np.round(np.real(sv_s.data), 4)}")
print("→ 全状態が等しい振幅 1/2 を持つ")
print()

# 次に H⊗² を |s⟩ にかける
qc2 = QuantumCircuit(2)
qc2.h([0, 1])  # |00⟩ → |s⟩
qc2.h([0, 1])  # |s⟩ → ?
sv_after = Statevector.from_instruction(qc2)
print(f"H⊗² 適用後: {np.round(np.real(sv_after.data), 4)}")
print("→ |s⟩ が |00⟩ に写る (H が基底変換として機能)")

## Slide 14 連動  |  Uncompute なし vs ありUncompute (アンシラを元に戻す操作) をしないと振幅増幅が崩れることを実験で確認する。- Uncompute あり: 解 |11⟩ がほぼ 100% で観測- Uncompute なし: アンシラに計算結果が残り、Diffusion の効果が打ち消される

In [ ]:
def grover_with_uncompute():
    """Uncompute あり (正しい実装) — N=4、解 |11⟩"""
    qc = QuantumCircuit(3, 2)
    qc.h([0, 1])
    qc.x(2); qc.h(2)  # アンシラ qubit 2 を |-⟩ に

    # Oracle (Toffoli は自身の逆なので Toffoli 1 回で uncompute も同時に行われる)
    qc.ccx(0, 1, 2)

    # Diffusion
    diffusion(qc, [0, 1])

    qc.measure([0, 1], [0, 1])
    return qc

def grover_without_uncompute():
    """Uncompute なし (壊れた実装) — アンシラに計算結果が残る"""
    qc = QuantumCircuit(4, 2)
    qc.h([0, 1])
    qc.x(2); qc.h(2)

    # アンシラ qubit 3 に AND を計算 → アンシラ qubit 2 に位相反転を伝播
    qc.ccx(0, 1, 3)   # aux に AND 結果を保存
    qc.cx(3, 2)        # aux → 位相反転アンシラ
    # ここで Uncompute をスキップ (aux qubit 3 に計算結果が残る)

    diffusion(qc, [0, 1])

    qc.measure([0, 1], [0, 1])
    return qc

counts_with = run_aer(grover_with_uncompute(), shots=2048)
counts_without = run_aer(grover_without_uncompute(), shots=2048)

print("Uncompute あり (正しい):")
for k in sorted(counts_with.keys()):
    print(f"  {k}: {counts_with[k]:>5}  ({counts_with[k]/2048*100:5.1f}%)")
print()
print("Uncompute なし (壊れた):")
for k in sorted(counts_without.keys()):
    print(f"  {k}: {counts_without[k]:>5}  ({counts_without[k]/2048*100:5.1f}%)")
print()

# 可視化
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, counts, title in [(axes[0], counts_with, 'Uncompute あり (正しい)'),
                           (axes[1], counts_without, 'Uncompute なし (壊れた)')]:
    keys = sorted(counts.keys())
    vals = [counts[k]/2048 for k in keys]
    colors = ['crimson' if k == '11' else 'steelblue' for k in keys]
    ax.bar(keys, vals, color=colors, edgecolor='black')
    ax.set_ylim(0, 1.05); ax.set_title(title); ax.set_ylabel('確率')
plt.tight_layout(); plt.show()

## Slide 18 連動  |  N=8, M=2 で「比は変えない」検証Slide 16 の「Oracle と Diffusion は解どうし/非解どうしの中の比を変えない」を実機検証する。- N=8, M=2 (解は |011⟩ と |101⟩ の 2 つ)- 各反復後に解 2 個の振幅、非解 6 個の振幅を取り出して比較- どの反復回数でも、解どうし・非解どうしの内部では振幅が等しいことを確認

In [ ]:
solutions_n8 = [0b011, 0b101]  # |011⟩=3, |101⟩=5
non_solutions_n8 = [i for i in range(8) if i not in solutions_n8]

def grover_oracle_n3(qc):
    """N=8 の Oracle: |011⟩ と |101⟩ に位相 -1"""
    # |011⟩
    qc.x(2)
    qc.h(2); qc.ccx(0, 1, 2); qc.h(2)
    qc.x(2)
    # |101⟩
    qc.x(1)
    qc.h(2); qc.ccx(0, 1, 2); qc.h(2)
    qc.x(1)

def grover_diffusion_n3(qc):
    """N=8 の Diffusion (平均反転)"""
    qc.h([0, 1, 2])
    qc.x([0, 1, 2])
    qc.h(2); qc.ccx(0, 1, 2); qc.h(2)
    qc.x([0, 1, 2])
    qc.h([0, 1, 2])
    qc.global_phase += np.pi

# 各反復後の振幅を取り出して、解と非解の中の比を確認
print("=== N=8, M=2: 解どうし / 非解どうしの中の比は変わらない ===\n")
for k in range(4):
    qc = QuantumCircuit(3)
    qc.h([0, 1, 2])  # 初期状態 |s⟩
    for _ in range(k):
        grover_oracle_n3(qc)
        grover_diffusion_n3(qc)
    sv = Statevector.from_instruction(qc)
    amps = np.real(sv.data)

    sol_amps = [amps[i] for i in solutions_n8]
    nonsol_amps = [amps[i] for i in non_solutions_n8]

    print(f"k={k}:")
    print(f"  解の振幅 (2 個):     {[round(a, 4) for a in sol_amps]}")
    print(f"    全部等しい: {np.allclose(sol_amps[0], sol_amps[1])}")
    print(f"  非解の振幅 (6 個):   {[round(a, 4) for a in nonsol_amps]}")
    print(f"    全部等しい: {np.allclose(nonsol_amps[0], nonsol_amps)}")
    print()

print("→ 反復ごとに、解 2 個は常に等しい振幅、非解 6 個も常に等しい振幅")

## Slide 19 連動  |  各反復 = 2θ 回転 の数値確認Slide 19 の主張「Grover 1 反復 = (|s'⟩ 軸からの) 角度を 2θ ずつ進める回転」を数値で確認する。N=8, M=2 のとき、θ = arcsin√(M/N) = arcsin√(1/4) = 30°。よって 1 反復ごとに角度が 2θ = 60° ずつ増えるはずである。各反復後の振幅から (|s'⟩ 軸からの) 角度を計算して、表として表示する。

In [ ]:
# N=8, M=2 で k=0..6 の角度を計算
# |w⟩ = (|011⟩ + |101⟩) / √2、|s'⟩ = 残り 6 状態の重ね合わせ / √6
# k 反復後、|s_k⟩ = sin((2k+1)θ)|w⟩ + cos((2k+1)θ)|s'⟩
# よって 角度 (|s'⟩ 軸からの)= arctan(α/β) = (2k+1)θ

theta_deg = np.degrees(np.arcsin(np.sqrt(2/8)))  # = 30°
print(f"θ = arcsin√(M/N) = arcsin√(2/8) = {theta_deg:.2f}°")
print(f"1 反復ごとに角度が 2θ = {2*theta_deg:.2f}° ずつ増えるはず\n")

# k=0..6 の角度を測定
angles = []
for k in range(7):
    qc = QuantumCircuit(3)
    qc.h([0, 1, 2])
    for _ in range(k):
        grover_oracle_n3(qc)
        grover_diffusion_n3(qc)
    sv = Statevector.from_instruction(qc)
    amps = np.real(sv.data)

    # |w⟩ 成分 (α) と |s'⟩ 成分 (β)
    alpha = np.sqrt(2) * amps[solutions_n8[0]]      # = sin((2k+1)θ)
    beta = np.sqrt(6) * amps[non_solutions_n8[0]]   # = cos((2k+1)θ)
    angle_deg = np.degrees(np.arctan2(alpha, beta))
    angles.append(angle_deg)

# np.arctan2 は (-180, 180] なので、k が進むと負の値に折り返す
# np.unwrap で連続な角度に直す
angles_unwrapped = np.degrees(np.unwrap(np.radians(angles)))

print("反復回数 k | 角度 | 期待値 (2k+1)·30°")
print("-" * 45)
for k, ang in enumerate(angles_unwrapped):
    expected = (2*k + 1) * theta_deg
    print(f"   k={k}      | {ang:6.2f}° | {expected:6.2f}°")

print("\n=== 各反復間の角度差 ===")
for i in range(1, len(angles_unwrapped)):
    diff = angles_unwrapped[i] - angles_unwrapped[i-1]
    print(f"  k={i-1} → k={i}: {diff:.2f}°")

print(f"\n→ 1 反復ごとに角度が {2*theta_deg:.2f}° (= 2θ) ずつ進む")

## Slide 20 連動  |  N=4 で k=0..7 の振動反復回数を変えると成功確率が振動することを確認する。N=4, M=1 のとき、θ = 30°、k_opt = 1。k=1 で 100% に達した後、k=2 では下がり、k=4 で再びピークになる (周期 3)。

In [ ]:
def grover_2qubit_with_k_iter(k):
    """2qubit Grover を k 回反復"""
    qc = QuantumCircuit(2, 2)
    qc.h([0, 1])
    for _ in range(k):
        qc.cz(0, 1)        # Oracle
        diffusion(qc, [0, 1])  # Diffusion
    qc.measure([0, 1], [0, 1])
    return qc

# k=0..7 の成功確率
ks = list(range(8))
success_probs = []
for k in ks:
    qc = grover_2qubit_with_k_iter(k)
    counts = run_aer(qc, shots=2048)
    p = counts.get('11', 0) / 2048
    success_probs.append(p)
    print(f"  k={k}: P(|11⟩) = {p:.3f}")

# 振動グラフ
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(ks, success_probs, 'o-', color='steelblue', markersize=8)
ax.axhline(1.0, color='gray', linestyle=':')
ax.set_xlabel('反復回数 k'); ax.set_ylabel('P(|11⟩)')
ax.set_ylim(-0.05, 1.10); ax.set_title('Grover 振動 (N=4)')
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Slide 21 連動  |  N=4 で k=2 を試す (k_opt 超過)k_opt = 1 を超えて k=2 にすると、振幅増幅が逆戻りする。

In [ ]:
qc_k2 = grover_2qubit_with_k_iter(2)
counts_k2 = run_aer(qc_k2, shots=2048)
print(f"k=2 の観測結果: {dict(sorted(counts_k2.items()))}")
print(f"→ |11⟩ の確率: {counts_k2.get('11', 0)/2048*100:.1f}%  (k=1 より下がる)")

## Slide 25 連動  |  ライツアウト古典解法Grover に入る前に、ライツアウトを古典で解く。3×3 ライツアウトはガウス消去法 (mod 2) で一瞬で解ける。

In [ ]:
def lights_out_classic(lights):
    A = np.zeros((9, 9), dtype=int)
    for i in range(9):
        r, c = i // 3, i % 3
        A[i][i] = 1
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < 3 and 0 <= nc < 3:
                A[i][nr * 3 + nc] = 1
    b = np.array(lights, dtype=int)
    M = np.hstack([A, b.reshape(-1, 1)])
    for col in range(9):
        for row in range(col, 9):
            if M[row][col] == 1:
                M[[col, row]] = M[[row, col]]
                break
        for row in range(9):
            if row != col and M[row][col] == 1:
                M[row] = (M[row] + M[col]) % 2
    return M[:, -1].tolist()

LIGHTS = [0, 1, 0, 1, 1, 0, 1, 1, 0]  # 例題盤面
solution = lights_out_classic(LIGHTS)
print(f"盤面 lights = {LIGHTS}")
print(f"古典解 solution = {solution}")
print(f"押すボタン: {[i for i, s in enumerate(solution) if s == 1]}")
# → solution = [0,1,0,1,0,1,0,0,1]、押すボタン: [1, 3, 5, 8]

## Slide 29 連動  |  flip_tile の構造確認 (デモ版)ボタン i を押すと、ボタン i 自身と上下左右の tile が反転する量子回路。ボタン 0 だけを押した場合の挙動を Statevector で確認する。期待される動き: tile[0], tile[1], tile[3] が反転 → tile = [1, 1, 0, 1, 0, 0, 0, 0, 0]

In [ ]:
def flip_tile_demo(qc, flip, tile, button_only):
    """指定ボタン 1 個だけが反転する場合 (デモ用)"""
    i = button_only
    r, c = i // 3, i % 3
    qc.cx(flip[i], tile[i])
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            qc.cx(flip[i], tile[nr * 3 + nc])

# ボタン 0 を押した時の挙動を確認
flip = QuantumRegister(9, 'flip')
tile = QuantumRegister(9, 'tile')
qc = QuantumCircuit(flip, tile)
qc.x(flip[0])  # flip[0] = 1 (ボタン 0 を押す)
flip_tile_demo(qc, flip, tile, 0)

sv = Statevector.from_instruction(qc)
probs = sv.probabilities([9, 10, 11, 12, 13, 14, 15, 16, 17])
top_idx = np.argmax(probs)
tile_state = format(top_idx, '09b')[::-1]
print(f"ボタン 0 を押した結果の tile 状態: {[int(b) for b in tile_state]}")
print(f"→ tile[0, 1, 3] が反転 (期待値 [1,1,0,1,0,0,0,0,0])")

## Slide 32 連動  |  flip_tile を実装する (7 分)ナイーブ版 Grover の心臓部 `flip_tile` を実装する。仕様: 9 個のボタンそれぞれについて、隣接関係に従って CNOT を並べる。隣接関係:- ボタン 0 を押す → tile[0, 1, 3] が反転- ボタン 1 を押す → tile[0, 1, 2, 4] が反転- ボタン 2 を押す → tile[1, 2, 5] が反転- ボタン 3 を押す → tile[0, 3, 4, 6] が反転- ボタン 4 を押す → tile[1, 3, 4, 5, 7] が反転- ボタン 5 を押す → tile[2, 4, 5, 8] が反転- ボタン 6 を押す → tile[3, 6, 7] が反転- ボタン 7 を押す → tile[4, 6, 7, 8] が反転- ボタン 8 を押す → tile[5, 7, 8] が反転

In [ ]:
def flip_tile(qc, flip, tile):
    """flip[i] が制御、tile[i] と隣接 tile が標的の CNOT を並べる

    例: ボタン 0 の場合
        qc.cx(flip[0], tile[0])
        qc.cx(flip[0], tile[1])
        qc.cx(flip[0], tile[3])

    全 9 ボタンについて同様に書く (隣接関係はマークダウン参照)
    """
    # ここに実装する
    pass


# 動作確認 (実装が正しければ、ボタン 0 単独で tile[0,1,3] が反転)
flip = QuantumRegister(9, 'flip')
tile = QuantumRegister(9, 'tile')
qc_test = QuantumCircuit(flip, tile)
qc_test.x(flip[0])  # ボタン 0 を押す
flip_tile(qc_test, flip, tile)

sv = Statevector.from_instruction(qc_test)
probs = sv.probabilities([9, 10, 11, 12, 13, 14, 15, 16, 17])
if np.max(probs) < 0.99:
    print("まだ未実装のようです (flip_tile が pass のまま)")
else:
    top_idx = np.argmax(probs)
    tile_state = [int(b) for b in format(top_idx, '09b')[::-1]]
    expected = [1, 1, 0, 1, 0, 0, 0, 0, 0]
    if tile_state == expected:
        print(f"ボタン 0 で tile = {tile_state} (正解)")
    else:
        print(f"ボタン 0 で tile = {tile_state}, 期待値 {expected}")

<details><summary>ヒント (詰まったら開く)</summary>**L1 (一般指針)**: 9 個のボタンそれぞれについて、隣接関係に従って CNOT を並べる。各ボタンは 3-5 個の `qc.cx` で実装可能。**L2 (構造ヒント)**:```python# 隣接関係を計算するヘルパーdef adjacent(i):    r, c = i // 3, i % 3    result = [i]  # 自分    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:        nr, nc = r + dr, c + dc        if 0 <= nr < 3 and 0 <= nc < 3:            result.append(nr * 3 + nc)    return resultdef flip_tile(qc, flip, tile):    for i in range(9):        for t in adjacent(i):            qc.cx(flip[i], tile[t])```**L3 (完成コード)**: 上記 L2 の実装を直接コピーすればよい。</details>

## Slide 34 連動  |  ナイーブ版 Grover 動作確認ナイーブ版 Grover ソルバを組み立てて動作確認する。### 26 qubit 版 (本筋)レジスタ flip(9) + tile(9) + oracle(1) + auxiliary(7) で構成。auxiliary 7 qubit を使った Toffoli ツリーで 9-qubit AND を段階的に計算する。構造が透明な代わりに AerSimulator では実行に数十分かかる。### 20 qubit 版 (高速)auxiliary 1 qubit + mcx 直接の構成。論理は 26 qubit 版と同じだが、シミュレータが内部で最適化を効かせるため約 12 秒で実行できる。講義中はこちらを使う。

In [ ]:
def lights_out_oracle_aux7(qc, tile, oracle, aux):
    """tile が全 0 (X 反転後は全 1) なら oracle に位相 -1
    auxiliary 7 qubit で 9-qubit AND を Toffoli ツリーで計算"""
    qc.x(tile)  # 全反転
    # 段階的 AND
    qc.ccx(tile[0], tile[1], aux[0])
    qc.ccx(aux[0], tile[2], aux[1])
    qc.ccx(aux[1], tile[3], aux[2])
    qc.ccx(aux[2], tile[4], aux[3])
    qc.ccx(aux[3], tile[5], aux[4])
    qc.ccx(aux[4], tile[6], aux[5])
    qc.ccx(aux[5], tile[7], aux[6])
    qc.ccx(aux[6], tile[8], oracle[0])  # 位相キックバック
    # Uncompute (逆順)
    qc.ccx(aux[5], tile[7], aux[6])
    qc.ccx(aux[4], tile[6], aux[5])
    qc.ccx(aux[3], tile[5], aux[4])
    qc.ccx(aux[2], tile[4], aux[3])
    qc.ccx(aux[1], tile[3], aux[2])
    qc.ccx(aux[0], tile[2], aux[1])
    qc.ccx(tile[0], tile[1], aux[0])
    qc.x(tile)  # 元に戻す

def naive_diffusion(qc, flip):
    """flip 9 qubit 上の標準 Diffusion"""
    qc.h(flip); qc.x(flip)
    qc.h(flip[8]); qc.mcx(list(flip[:8]), flip[8]); qc.h(flip[8])
    qc.x(flip); qc.h(flip)
    qc.global_phase += np.pi

def build_naive_grover(lights, num_iterations):
    """ナイーブ版 Grover ソルバ (26 qubit、本筋)"""
    flip = QuantumRegister(9, 'flip')
    tile = QuantumRegister(9, 'tile')
    oracle = QuantumRegister(1, 'oracle')
    aux = QuantumRegister(7, 'aux')
    cr = ClassicalRegister(9, 'c')
    qc = QuantumCircuit(flip, tile, oracle, aux, cr)

    # 初期化
    qc.h(flip)
    for i, l in enumerate(lights):
        if l == 1: qc.x(tile[i])
    qc.x(oracle[0]); qc.h(oracle[0])

    # k_opt 反復
    for _ in range(num_iterations):
        flip_tile(qc, flip, tile)
        lights_out_oracle_aux7(qc, tile, oracle, aux)
        flip_tile(qc, flip, tile)
        naive_diffusion(qc, flip)

    qc.measure(flip, cr)
    return qc

print("build_naive_grover() 定義完了 (26 qubit 版、AerSimulator では数十分)")

In [ ]:
def lights_out_oracle_fast(qc, tile, oracle, aux):
    """高速版: mcx を直接使う (Toffoli ツリーは内部最適化に任せる)"""
    qc.x(tile)
    qc.mcx(list(tile), aux[0])  # 9-qubit AND を直接
    qc.cx(aux[0], oracle[0])     # 位相キックバック
    qc.mcx(list(tile), aux[0])  # Uncompute
    qc.x(tile)

def build_naive_grover_fast(lights, num_iterations):
    """高速版 (20 qubit、AerSimulator で約 12 秒)"""
    flip = QuantumRegister(9, 'flip')
    tile = QuantumRegister(9, 'tile')
    oracle = QuantumRegister(1, 'oracle')
    aux = QuantumRegister(1, 'aux')   # ここだけ違う (7 → 1)
    cr = ClassicalRegister(9, 'c')
    qc = QuantumCircuit(flip, tile, oracle, aux, cr)

    qc.h(flip)
    for i, l in enumerate(lights):
        if l == 1: qc.x(tile[i])
    qc.x(oracle[0]); qc.h(oracle[0])

    for _ in range(num_iterations):
        flip_tile(qc, flip, tile)
        lights_out_oracle_fast(qc, tile, oracle, aux)
        flip_tile(qc, flip, tile)
        naive_diffusion(qc, flip)

    qc.measure(flip, cr)
    return qc

# 高速版で実行
import time
print("AerSimulator で k=17 / 20 qubit 版を実行中...")
qc_fast = build_naive_grover_fast(LIGHTS, num_iterations=17)
print(f"  qubit 数: {qc_fast.num_qubits}, ゲート数: {len(qc_fast.data)}")
t0 = time.time()
counts = run_aer(qc_fast, shots=2048)
print(f"  実行時間: {time.time()-t0:.1f} 秒")
top = sorted(counts.items(), key=lambda x: -x[1])[:5]
print(f"  Top 5 観測:")
for s, c in top:
    print(f"    {s}: {c} ({c/2048*100:.2f}%)")
print()
print(f"古典解と一致確認:")
print(f"  古典解 solution = {solution}")
print(f"  押すボタン:    {[i for i, x in enumerate(solution) if x == 1]}")

In [ ]:
# ヒストグラム可視化
fig, ax = plt.subplots(figsize=(11, 4))
top10 = sorted(counts.items(), key=lambda x: -x[1])[:10]
labels = [s for s, _ in top10]
values = [c/2048 for _, c in top10]
colors = ['crimson' if i == 0 else 'steelblue' for i in range(len(labels))]
ax.bar(range(len(labels)), values, color=colors, edgecolor='black')
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right', family='monospace', fontsize=10)
ax.set_ylabel('確率')
ax.set_title('ナイーブ版 Grover の観測結果 (k_opt=17、shots=2048)')
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

## 中間レポート オプション 1+ 用スケルトン (持ち帰り)ナイーブ版を改良するためのスケルトン。中間レポートでこれを完成させる。ヒント:- 行依存性: 1 行目を決めれば 2 行目以降は一意に決まる → flip 9 qubit のうち 3 qubit のみ重ね合わせ- 列依存性 / 対称性: 自分で発見してみる- Diffusion 別実装 / Oracle 構造最適化: 自分で発見してみる最低条件:- ナイーブ版より少ない qubit 数で完成- 動作確認 (ヒストグラムで解の振幅が突出)

In [ ]:
# 行依存性ヒント:
#   1 行目 (flip[0,1,2]) を決めれば、
#   2 行目 (flip[3,4,5]) は 1 行目の tile 状態から一意に決まる。
#   3 行目 (flip[6,7,8]) も同様。
#   → flip 9 qubit のうち、本当に重ね合わせる必要があるのは 3 qubit (1 行目) のみ。

def flip_1(qc, flip, tile):
    """1 行目のボタンを押す操作 (flip[0,1,2] のみ)"""
    pass

def inv_1(qc, flip, tile):
    """1 行目の状態を見て、2 行目を一意決定"""
    pass

def flip_2(qc, flip, tile):
    """2 行目のボタンを押す操作 (flip[3,4,5])"""
    pass

def inv_2(qc, flip, tile):
    """2 行目の tile を見て、3 行目を一意決定"""
    pass

def flip_3(qc, flip, tile):
    """3 行目のボタンを押す操作 (flip[6,7,8])"""
    pass

def improved_diffusion(qc, flip):
    """flip[0:3] の 3 qubit 空間で標準 Diffusion"""
    pass

def build_improved_grover(lights, num_iterations):
    """改良版 Grover ソルバ
    flip(9) + tile(9) + oracle(1) + aux(1) = 20 qubit

    重ね合わせは flip[0:3] の 3 qubit のみ → 探索空間 2^3 = 8
    k_opt = ⌊π/(4·arcsin√(1/8))⌋ = 2
    """
    pass


print("改良版実装は中間レポート オプション 1+ で取り組む")
print(f"  ヒント: 探索空間 2⁹=512 → 2³=8 削減なら k_opt も大きく減る")
print(f"  理論値: ⌊π/(4·arcsin√(1/8))⌋ ≈ 2")
print(f"  ナイーブ版 26 qubit / k=17 → 改良版 20 qubit / k=2 が目標")

## 持ち帰り課題 (任意)中間レポートに取り組む前のウォームアップ。

In [ ]:
# 課題 A: 異なる初期盤面で動作確認
# lights を変えても解けるか? 高速版でいくつか試す
lights_other = [1, 0, 1, 0, 1, 0, 1, 0, 1]  # 別の盤面例
solution_other = lights_out_classic(lights_other)
print(f"別盤面 lights = {lights_other}")
print(f"古典解 = {solution_other}")

# Grover でも解けるか確認
qc_other = build_naive_grover_fast(lights_other, num_iterations=17)
counts_other = run_aer(qc_other, shots=1024)
top = sorted(counts_other.items(), key=lambda x: -x[1])[:3]
print(f"Grover Top 3 観測:")
for s, c in top:
    print(f"  {s}: {c} ({c/1024*100:.1f}%)")

In [ ]:
# 課題 B: k_opt の影響を実験 (k=10/13/17/20 で成功確率を比較)
results_k = {}
for k in [5, 10, 13, 17, 20]:
    qc = build_naive_grover_fast(LIGHTS, num_iterations=k)
    cnts = run_aer(qc, shots=512)
    sol_str = ''.join(str(b) for b in reversed(solution))
    p = cnts.get(sol_str, 0) / 512
    results_k[k] = p
    print(f"  k={k}: P(success) = {p:.3f}")

# 比較プロット
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar([str(k) for k in results_k.keys()], list(results_k.values()),
       color=['crimson' if k == 17 else 'steelblue' for k in results_k.keys()],
       edgecolor='black')
ax.set_xlabel('反復回数 k'); ax.set_ylabel('P(success)')
ax.set_ylim(0, 1.05); ax.set_title('反復回数の影響 (3x3 Lights-Out)')
ax.grid(axis='y', alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# 課題 C: 改良版に挑戦 (中間レポート オプション 1+)
# 上の build_improved_grover スケルトンを完成させる

# 行依存性のヒント:
#   1. flip_1 で 1 行目 (flip[0..2]) のボタンを押す
#   2. inv_1 で 1 行目 tile を見て 2 行目 (flip[3..5]) を決める
#   3. flip_2 で 2 行目を押す
#   4. inv_2 で 3 行目 (flip[6..8]) を決める
#   5. flip_3 で 3 行目を押す
#   6. lights_out_oracle で全消灯チェック
#   7. U† (逆順 Uncompute)
#   8. improved_diffusion (flip[0..2] の 3 qubit 空間で)

print("中間レポート オプション 1+ で挑戦 → 動作したら 20 qubit / k≈2 で 99% を目指す")